# PPO 在 LLM 中的最小训练实现

配套知识点：`common_knowledge/14_ppo.md`。这里用一个**字符级 tiny-LM** 跑通 RLHF-PPO 的完整回路，把笔记里的每个部件落到代码：

- **四模型**：Policy(Actor) + Value(Critic) 共享 backbone；Reference(冻结) 算 KL；这里用**规则奖励**代替独立 RM（可验证任务，最省事）。
- **玩具任务**：词表是数字 `0-9`，奖励 = 生成序列里**偶数字符的个数**（越多越好）。训练目标是让策略学会多输出偶数字 —— 一个有客观对错的可验证奖励（RLVR 味道）。
- **回路**：`rollout(旧策略采样) → 逐token奖励(RM分数末token + 逐token KL惩罚) → GAE → PPO clip 更新(多 epoch)`。

跑起来能看到 **平均奖励随训练上升**，直观感受 clip、advantage、KL 的作用。


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

VOCAB = 10          # 数字 0-9
GEN_LEN = 12        # 每条生成序列长度
BOS = 0             # 用 0 当起始 token（简化）
print(device)


cpu


## 1. Policy + Value：共享 backbone 的 actor-critic

一个 tiny Transformer decoder，接两个头：`lm_head`（策略 logits）和 `value_head`（Critic 估 V(s)）。这对应笔记 §五。

In [ ]:
class TinyLM(nn.Module):
    def __init__(self, vocab=VOCAB, d=64, nhead=4, nlayer=2, maxlen=64):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(maxlen, d)
        layer = nn.TransformerEncoderLayer(d, nhead, 4*d, batch_first=True, dropout=0.0)
        self.blocks = nn.TransformerEncoder(layer, nlayer)
        self.ln = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab)     # actor: 下一 token 分布
        self.value_head = nn.Linear(d, 1)      # critic: V(s)

    def forward(self, ids):
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        h = self.tok(ids) + self.pos(pos)
        mask = torch.triu(torch.ones(T, T, device=ids.device), diagonal=1).bool()  # 因果掩码
        h = self.ln(self.blocks(h, mask=mask))
        logits = self.lm_head(h)               # [B,T,vocab]
        value = self.value_head(h).squeeze(-1) # [B,T]
        return logits, value

policy = TinyLM().to(device)
ref = TinyLM().to(device)
ref.load_state_dict(policy.state_dict())       # Reference = 初始策略快照（冻结）
for p in ref.parameters():
    p.requires_grad_(False)

opt = torch.optim.Adam(policy.parameters(), lr=3e-4)
print("params:", sum(p.numel() for p in policy.parameters()))


## 2. 规则奖励（代替 RM）

可验证奖励：数一条序列里有几个偶数字。对应笔记 §6.1 里的「RM 分数」，只是这里用规则算、只在**整条序列**上给一个标量。

In [ ]:
def reward_fn(seq):
    # seq: [B, GEN_LEN] 生成的 token（不含 BOS）。奖励 = 偶数字个数
    even = ((seq % 2) == 0).float().sum(dim=-1)   # [B]
    return even   # 范围 0..GEN_LEN


## 3. Rollout：用旧策略采样

采样 = RL 的「rollout」。记录每个 token 在**旧策略**下的 log-prob（重要性采样的分母，笔记 §三）。

In [ ]:
@torch.no_grad()
def rollout(model, B=64):
    ids = torch.full((B, 1), BOS, dtype=torch.long, device=device)
    old_logps = []
    for _ in range(GEN_LEN):
        logits, _ = model(ids)
        dist = torch.distributions.Categorical(logits=logits[:, -1])
        a = dist.sample()
        old_logps.append(dist.log_prob(a))
        ids = torch.cat([ids, a.unsqueeze(1)], dim=1)
    seq = ids[:, 1:]                        # 去掉 BOS -> [B, GEN_LEN]
    old_logp = torch.stack(old_logps, dim=1)  # [B, GEN_LEN]
    return ids, seq, old_logp


## 4. 逐 token 奖励 + GAE

- **逐 token 奖励**（笔记 §6.1）：`r_t = RM分数·(末token) − β·KL_t`。KL 用 `logπ_θ − logπ_ref` 逐 token 算。
- **GAE**（笔记 §四）：`δ_t = r_t + γV(s_{t+1}) − V(s_t)`，`Â_t = Σ (γλ)^l δ_{t+l}`。这里序列级 bandit，`γ=1`。


In [ ]:
GAMMA, LAM, BETA = 1.0, 0.95, 0.02

def token_logp_and_value(model, ids):
    # 返回生成段每个位置的 log-prob(在已采样 token 上) 和 value
    logits, value = model(ids)             # 用 ids[:, :-1] 预测 ids[:, 1:]
    logp_all = F.log_softmax(logits[:, :-1], dim=-1)
    targets = ids[:, 1:]                   # [B, GEN_LEN]
    logp = logp_all.gather(-1, targets.unsqueeze(-1)).squeeze(-1)  # [B, GEN_LEN]
    v = value[:, :-1]                      # V(s_t)，对齐生成段 [B, GEN_LEN]
    return logp, v

def compute_gae(rewards, values):
    # rewards, values: [B, T]。values 是 V(s_t)；末步之后 bootstrap=0
    B, T = rewards.shape
    adv = torch.zeros_like(rewards)
    last = torch.zeros(B, device=rewards.device)
    for t in reversed(range(T)):
        v_next = values[:, t+1] if t+1 < T else torch.zeros(B, device=rewards.device)
        delta = rewards[:, t] + GAMMA * v_next - values[:, t]
        last = delta + GAMMA * LAM * last
        adv[:, t] = last
    returns = adv + values
    return adv, returns


## 5. PPO 更新：clip 目标 + value loss + entropy

同一批 rollout 数据跑多个 epoch（重要性采样让旧数据可复用，笔记 §三）。核心是 clip 目标（笔记 §二）：

`L = min(ρ·Â, clip(ρ,1-ε,1+ε)·Â)`，`ρ = exp(logπ_θ − logπ_old)`。


In [ ]:
EPS, PPO_EPOCHS, VF_COEF, ENT_COEF = 0.2, 4, 0.5, 0.01

def ppo_update(ids, seq, old_logp):
    with torch.no_grad():
        ref_logp, _ = token_logp_and_value(ref, ids)     # 参考策略 log-prob
        rm = reward_fn(seq)                                # [B] 整条序列 RM 分数
        # 逐 token 奖励：KL 惩罚逐 token，RM 分数只加在末 token
        _, v_old = token_logp_and_value(policy, ids)
        kl = old_logp - ref_logp                           # logπ_old - logπ_ref ≈ 逐token KL
        rewards = -BETA * kl
        rewards[:, -1] += rm                               # 末 token 加 RM 分数
        adv, returns = compute_gae(rewards, v_old)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)      # 优势归一化（37 details）

    stats = {}
    for _ in range(PPO_EPOCHS):
        logp, v = token_logp_and_value(policy, ids)
        ratio = torch.exp(logp - old_logp)                 # ρ
        unclipped = ratio * adv
        clipped = torch.clamp(ratio, 1-EPS, 1+EPS) * adv
        pg_loss = -torch.min(unclipped, clipped).mean()    # clip 目标（取负因为要最大化）
        v_loss = F.mse_loss(v, returns)                    # Critic 回归 return
        # entropy bonus 鼓励探索
        logits, _ = policy(ids)
        ent = torch.distributions.Categorical(logits=logits[:, :-1]).entropy().mean()
        loss = pg_loss + VF_COEF * v_loss - ENT_COEF * ent

        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        opt.step()
        stats = {"pg": pg_loss.item(), "vf": v_loss.item(), "ent": ent.item(),
                 "clipfrac": ((ratio-1).abs() > EPS).float().mean().item()}
    return rm.mean().item(), stats


## 6. 训练：看平均奖励爬升

每步：旧策略 rollout → PPO 多 epoch 更新。奖励上限是 `GEN_LEN=12`（全偶数）。

In [ ]:
for step in range(150):
    ids, seq, old_logp = rollout(policy, B=128)
    mean_r, st = ppo_update(ids, seq, old_logp)
    if step % 15 == 0 or step == 149:
        print(f"step {step:3d} | avg_reward {mean_r:5.2f}/{GEN_LEN} "
              f"| pg {st['pg']:+.3f} vf {st['vf']:.3f} ent {st['ent']:.3f} clipfrac {st['clipfrac']:.2f}")


## 7. 对照笔记：代码里的部件 ↔ §号

| 代码 | 笔记 | 说明 |
|---|---|---|
| `TinyLM` 的 `lm_head`/`value_head` | §五 | actor-critic 共享 backbone |
| `ref` 冻结副本 | §六 ④ Reference | 算 KL 防跑偏 |
| `reward_fn` | §六 ③ RM | 这里用规则奖励代替独立 RM |
| `rewards[:,-1] += rm; rewards = -β·kl` | §6.1 | RM 分数只在末 token，KL 逐 token |
| `compute_gae` | §四 | GAE，λ 在偏差-方差间插值 |
| `torch.min(unclipped, clipped)` | §二 | clip 目标外层取 min |
| `clipfrac` | §二 | 有多少 token 触发裁剪（监控更新幅度） |

**动手体验**：把 `EPS` 调到很大（如 5.0）≈ 关掉 clip → 观察训练是否更容易震荡/崩；把 `BETA` 调到很大 → KL 惩罚过强，奖励涨不上去（策略被死死拴在 ref 附近）。这正对应笔记 §六「KL 太强 ≈ 拴住策略」与 §二「clip 防更新过头」。

**对照 GRPO（§七）**：若把 `value_head`/`compute_gae` 删掉，改用「同一 prompt 采一组、组内均值当 baseline」，就变成 GRPO —— 少的正是这里的 Critic。见 `code/` 后续 GRPO notebook。
